# TASK3 Step 1 — E0 adapter + BM25 + fixed decoding
Private Kaggle GPU only. Stop immediately on any hash mismatch. This notebook does not train, submit, or promote.

In [ ]:
!pip install -q -e '/kaggle/input/legalqa-code-v3[generation,eval]'

In [ ]:
import json
import os
from pathlib import Path

import torch

from legalqa.core.config import load_config
from legalqa.core.manifest import adapter_sha256, code_hash, config_hash, sha256_file

CODE = Path('/kaggle/input/legalqa-code-v3')
DATA = Path('/kaggle/input/task2-data-v3')
INDEX = Path('/kaggle/input/task2-index-v3')
ADAPTER = Path('/kaggle/input/e0-adapter-v1')
WORK = Path('/kaggle/working/legalqa-runtime')
INDEX_MOUNT = WORK / 'Data/Task2/indexes/task2-data-v3-bm25-candidate/index'
WORK.mkdir(parents=True, exist_ok=True)
INDEX_MOUNT.parent.mkdir(parents=True, exist_ok=True)
(WORK / 'models').mkdir(exist_ok=True)
os.symlink(INDEX, INDEX_MOUNT, target_is_directory=True)
os.symlink(ADAPTER, WORK / 'models/e0-full-v1', target_is_directory=True)
os.chdir(WORK)

expected = json.loads((CODE / 'expected_hashes_step1.json').read_text())
config = load_config(CODE / 'configs/e1_rag_step1.yaml')
assert torch.cuda.is_available(), 'NO_CUDA_DEVICE'
assert code_hash() == expected['code_hash'], 'CODE_HASH_MISMATCH'
assert config_hash(config) == expected['config_hash'], 'CONFIG_HASH_MISMATCH'
assert sha256_file(DATA / 'manifest.json') == expected['release_manifest_sha256'], 'RELEASE_HASH_MISMATCH'
assert sha256_file(DATA / 'qa/validation.jsonl') == expected['input_sha256'], 'INPUT_HASH_MISMATCH'
assert sha256_file(INDEX / 'index_manifest.json') == expected['index_manifest_sha256'], 'INDEX_MANIFEST_HASH_MISMATCH'
assert sha256_file(INDEX / 'bm25.sqlite3') == expected['index_artifact_sha256'], 'INDEX_ARTIFACT_HASH_MISMATCH'
assert adapter_sha256(ADAPTER) == expected['adapter_sha256'], 'ADAPTER_HASH_MISMATCH'
os.environ['LEGALQA_GIT_COMMIT'] = expected['git_commit']

In [ ]:
from legalqa.generation.pipeline import LegalQAPipeline

predictions = Path('/kaggle/working/step1_validation_predictions.json')
pipeline = LegalQAPipeline(config)
report = pipeline.predict_file(
    input_path=DATA / 'qa/validation.jsonl',
    output_path=predictions,
    trace_path='/kaggle/working/step1_trace.json',
    progress=print,
    environment='kaggle_gpu',
    adapter_path=ADAPTER,
)
report

In [ ]:
from legalqa.core.io import write_json
from legalqa.evaluation import evaluate_qa_files

metrics = evaluate_qa_files(predictions, DATA / 'qa/validation.jsonl')
write_json('/kaggle/working/step1_metrics.json', metrics)
metrics